In [ ]:
# ============================================================================
# 00_run_log_init
# ----------------------------------------------------------------------------
# Stage 0 of the Airflow DAG (task `init_run_log`). Idempotent setup of the two
# bookkeeping tables in lh_synthea_gold. Safe to run on every DAG trigger
# (CREATE ... IF NOT EXISTS):
#
#   agent.run_log     -- audit trail written by the CEO Insights agent (carried
#                        over unchanged from synthea-data-ws).
#   control.run_state -- NEW for the Option-B port: per-(run_id_root, cohort_id,
#                        stage) execution state that drives skip/restart.
#
# This is the only additive change to the source notebook: the control.run_state
# DDL (see ../sql/control_run_state.sql) is wired in here.
# ============================================================================

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# --- agent.run_log (unchanged from synthea-data-ws) ------------------------
spark.sql("CREATE SCHEMA IF NOT EXISTS lh_synthea_gold.agent")

spark.sql("""
CREATE TABLE IF NOT EXISTS lh_synthea_gold.agent.run_log (
    run_id          STRING       NOT NULL,
    agent_name      STRING       NOT NULL,
    started_at      TIMESTAMP    NOT NULL,
    finished_at     TIMESTAMP,
    prompt_summary  STRING,
    tools_called    ARRAY<STRING>,
    output_summary  STRING,
    recipients      ARRAY<STRING>,
    status          STRING       NOT NULL,
    error           STRING
)
USING DELTA
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true'
)
""")

# --- control.run_state (NEW: per-cohort skip/restart) ----------------------
spark.sql("CREATE SCHEMA IF NOT EXISTS lh_synthea_gold.control")

spark.sql("""
CREATE TABLE IF NOT EXISTS lh_synthea_gold.control.run_state (
    run_id_root   STRING  COMMENT 'Run grouping key (utcnow yyyyMMddHHmmss)',
    cohort_id     STRING  COMMENT 'Cohort = dataset_id; "ALL" for whole-run stages',
    stage         STRING  COMMENT 'generate | bronze_to_silver | silver_to_gold',
    status        STRING  COMMENT 'RUNNING | SUCCEEDED | FAILED',
    attempt       INT     COMMENT 'Incrementing attempt counter',
    started_ts    TIMESTAMP,
    ended_ts      TIMESTAMP,
    error         STRING  COMMENT 'Truncated error text when status=FAILED'
)
USING DELTA
""")

print(spark.sql("DESCRIBE TABLE lh_synthea_gold.agent.run_log").toPandas().to_string())
print(spark.sql("DESCRIBE TABLE lh_synthea_gold.control.run_state").toPandas().to_string())

mssparkutils.notebook.exit({
    "status": "ok",
    "tables": ["lh_synthea_gold.agent.run_log", "lh_synthea_gold.control.run_state"],
})
